# AnytimeYOLO — Fine-tune GELAN-m with Early Exits on COCO

Flow: **Clone repo → pip install → download GELAN-m weights → fine-tune exits → push best to HF Hub**

In [ ]:
# ── 1. Clone repo ──────────────────────────────────────────────────────────
!rm -rf /content/YoloGelanv9EarlyExitTrain
!git clone https://github.com/airlanggawicaksono/YoloGelanv9EarlyExitTrain.git /content/YoloGelanv9EarlyExitTrain

In [ ]:
# ── 2. Install deps ────────────────────────────────────────────────────────
!pip install -q -r /content/YoloGelanv9EarlyExitTrain/model/yolov9/requirements.txt
!pip install -q -U huggingface_hub wandb

import torch
print(f'torch {torch.__version__}  |  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

In [ ]:
# ── 3. Paths ───────────────────────────────────────────────────────────────
import sys
from pathlib import Path

PROJECT_ROOT = Path('/content/YoloGelanv9EarlyExitTrain')
YOLOV9_ROOT  = PROJECT_ROOT / 'model' / 'yolov9'
EE_CFG       = PROJECT_ROOT / 'early_exit' / 'configs' / 'gelan-m-ee.yaml'
HYP_YAML     = YOLOV9_ROOT  / 'data' / 'hyps' / 'hyp.scratch-high.yaml'
RUNS_DIR     = PROJECT_ROOT / 'runs' / 'train'
COCO_YAML    = None   # set by COCO cell after Roboflow download

sys.path.insert(0, str(YOLOV9_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

assert EE_CFG.exists(),   f'missing {EE_CFG}'
assert HYP_YAML.exists(), f'missing {HYP_YAML}'
print('paths OK')

In [ ]:
# ── 4. Credentials ─────────────────────────────────────────────────────────
# Set HF_TOKEN and RF_API_KEY in Colab Secrets (key icon, left sidebar).
from google.colab import userdata
from huggingface_hub import login, whoami

HF_TOKEN   = userdata.get('HF_TOKEN')
RF_API_KEY = userdata.get('RF_API_KEY')
HF_REPO    = 'wicaksonolxn/gelan-m-anytime-yolo'

login(token=HF_TOKEN, add_to_git_credential=False)
print('Logged in as:', whoami().get('name', 'unknown'))

In [ ]:
# ── 5. Download COCO via Roboflow + cache to Drive ─────────────────────────
import os, yaml
from google.colab import drive

drive.mount('/content/drive')

COCO_DRIVE = Path('/content/drive/MyDrive/datasets/coco-rf')
COCO_LOCAL = PROJECT_ROOT / 'datasets' / 'coco'
COCO_LOCAL.parent.mkdir(parents=True, exist_ok=True)

if not COCO_DRIVE.exists():
    !pip install -q roboflow
    from roboflow import Roboflow
    rf      = Roboflow(api_key=RF_API_KEY)
    dataset = rf.workspace("microsoft").project("coco").version(18).download(
        "yolov9", location=str(COCO_DRIVE)
    )
    print('Download complete.')
else:
    print('COCO found on Drive — skipping download.')

if not COCO_LOCAL.exists():
    os.symlink(str(COCO_DRIVE), str(COCO_LOCAL))
    print(f'Symlinked {COCO_LOCAL} -> {COCO_DRIVE}')

# patch yaml: absolute paths, use test split as val (version 18 has no valid/)
_yaml_path = COCO_DRIVE / 'data.yaml'
with open(_yaml_path) as f:
    _cfg = yaml.safe_load(f)
_cfg['path']  = str(COCO_DRIVE)
_cfg['train'] = str(COCO_DRIVE / 'train' / 'images')
_cfg['val']   = str(COCO_DRIVE / 'test'  / 'images')
_cfg['test']  = str(COCO_DRIVE / 'test'  / 'images')
with open(_yaml_path, 'w') as f:
    yaml.dump(_cfg, f)

COCO_YAML = COCO_LOCAL / 'data.yaml'
assert COCO_YAML.exists(), f'data.yaml missing at {COCO_YAML}'
print('COCO ready. yaml:', COCO_YAML)

In [ ]:
# ── 6. Pretrained GELAN-m weights ──────────────────────────────────────────
PRETRAIN_WEIGHTS = YOLOV9_ROOT / 'gelan-m.pt'

if not PRETRAIN_WEIGHTS.exists():
    !wget -c https://github.com/WongKinYiu/yolov9/releases/download/v0.1/gelan-m.pt \
          -O {PRETRAIN_WEIGHTS}

assert PRETRAIN_WEIGHTS.exists(), 'gelan-m.pt not found'
print('weights:', PRETRAIN_WEIGHTS)

In [ ]:
# ── 7. Fine-tune with early exits ─────────────────────────────────────────
import sys, os, torch, functools
os.chdir(str(YOLOV9_ROOT))
sys.path.insert(0, str(YOLOV9_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

# PyTorch 2.6+ defaults weights_only=True — breaks yolov9 checkpoint loading.
# functools.wraps sets __wrapped__, so re-running this cell skips re-patching.
if not hasattr(torch.load, '__wrapped__'):
    _orig_load = torch.load
    @functools.wraps(_orig_load)
    def _load(*a, **kw):
        kw.setdefault('weights_only', False)
        return _orig_load(*a, **kw)
    torch.load = _load

import train_triple
from early_exit.model import EarlyExitModel
from early_exit.loss  import EarlyExitLoss

train_triple.Model = EarlyExitModel
train_triple.ComputeLoss = lambda model, use_dfl=True: EarlyExitLoss(
    model, use_dfl, sample_exits=True
)

train_triple.run(
    cfg        = str(EE_CFG),
    data       = str(COCO_YAML),
    hyp        = str(HYP_YAML),
    weights    = str(PRETRAIN_WEIGHTS),
    epochs     = 300,
    batch_size = 32,
    imgsz      = 640,
    optimizer  = 'SGD',
    device     = '0',
    workers    = 4,
    project    = str(RUNS_DIR),
    name       = 'gelan-m-ee',
    close_mosaic = 10,
    cos_lr     = False,
    patience   = 100,
    save_period = 50,
)

os.chdir(str(PROJECT_ROOT))

In [ ]:
# ── 8. Push best checkpoint to HF Hub ─────────────────────────────────────
import glob, pandas as pd
from huggingface_hub import HfApi, create_repo

# find actual run dir (increment_path may have made gelan-m-ee2 etc.)
run_dirs = sorted(glob.glob(str(RUNS_DIR / 'gelan-m-ee*')))
assert run_dirs, f'no run dir found under {RUNS_DIR}'
SAVE_DIR = Path(run_dirs[-1])
BEST_PT  = SAVE_DIR / 'weights' / 'best.pt'
LAST_PT  = SAVE_DIR / 'weights' / 'last.pt'
RESULTS  = SAVE_DIR / 'results.csv'
print('Run dir:', SAVE_DIR)

assert BEST_PT.exists(), 'best.pt not found — training may have failed'

def _show_best(results_path):
    df = pd.read_csv(results_path)
    df.columns = df.columns.str.strip()
    map_col   = next((c for c in df.columns if 'mAP_0.5' in c and '0.95' not in c), None)
    map95_col = next((c for c in df.columns if 'mAP_0.5:0.95' in c), None)
    if map_col is None:
        print('mAP_0.5 column not found in results.csv')
        return
    best = df.loc[df[map_col].idxmax()]
    print(f'Best epoch : {int(best["epoch"])}')
    print(f'mAP@0.5   : {best[map_col]:.4f}')
    if map95_col:
        print(f'mAP@0.5:95: {best[map95_col]:.4f}')

if RESULTS.exists():
    _show_best(RESULTS)

api = HfApi(token=HF_TOKEN)
create_repo(HF_REPO, exist_ok=True, repo_type='model', token=HF_TOKEN)

for local, remote in [
    (BEST_PT, 'best.pt'),
    (LAST_PT, 'last.pt'),
    (EE_CFG,  'gelan-m-ee.yaml'),
    (RESULTS, 'results.csv'),
]:
    if Path(local).exists():
        api.upload_file(path_or_fileobj=str(local), path_in_repo=remote, repo_id=HF_REPO)
        print(f'uploaded {remote}')

print(f'\nPushed to: https://huggingface.co/{HF_REPO}')